In [ ]:
# =========================
# 1. Install deps
# =========================
!pip -q install pandas scikit-learn numpy matplotlib


In [ ]:
# =========================
# 2. Imports
# =========================
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import normalize


In [ ]:
# =========================
# 0. Public dataset -> /content/corpus.csv
# =========================
from sklearn.datasets import fetch_20newsgroups
import pandas as pd
import re
from pathlib import Path

# Завантажуємо raw texts
train = fetch_20newsgroups(subset="train", remove=())
test = fetch_20newsgroups(subset="test", remove=())

texts = list(train.data) + list(test.data)

def clean_text(t: str) -> str:
    t = t or ""
    t = re.sub(r"\S+@\S+", " ", t)          # emails
    t = re.sub(r"http\S+|www\.\S+", " ", t)  # urls
    t = re.sub(r"[^A-Za-zÀ-ÿ0-9\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip().lower()
    return t

df = pd.DataFrame({
    "doc_id": range(len(texts)),
    "raw_text": texts,
})
df["processed_v2"] = df["raw_text"].map(clean_text)

out_path = Path("/content/corpus.csv")
df[["doc_id", "processed_v2"]].to_csv(out_path, index=False)

print("Saved:", out_path)
print("Rows:", len(df))
print(df.head(3))

Saved: /content/corpus.csv
Rows: 18846
   doc_id                                           raw_text  \
0       0  From: lerxst@wam.umd.edu (where's my thing)\nS...   
1       1  From: guykuo@carson.u.washington.edu (Guy Kuo)...   
2       2  From: twillis@ec.ecn.purdue.edu (Thomas E Will...   

                                        processed_v2  
0  from where s my thing subject what car is this...  
1  from guy kuo subject si clock poll final call ...  
2  from thomas e willis subject pb questions orga...  


In [ ]:
# =========================
# 3. Config
# =========================
DATA_PATH = "/content/corpus.csv"
TEXT_COL = "processed_v2"
ID_COL = None

# Topic modeling settings
LSA_K_LIST = [5, 8]
LDA_K_LIST = [5, 8]

# Vectorizer settings
MIN_DF = 3
MAX_DF = 0.90
NGRAM_RANGE_LSA = (1, 2)
NGRAM_RANGE_LDA = (1, 1)

# Document filtering
MIN_TOKEN_LEN = 20   # very short docs often hurt topic quality
MIN_WORDS = 5

OUT_DIR = Path("/content/lab8_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 200)


In [ ]:
# =========================
# 4. Load data
# =========================
df = pd.read_csv(DATA_PATH)
print("Columns:", list(df.columns))
print("Rows before filtering:", len(df))

if TEXT_COL not in df.columns:
    raise ValueError(f"TEXT_COL='{TEXT_COL}' not found in CSV columns")

if ID_COL is not None and ID_COL not in df.columns:
    raise ValueError(f"ID_COL='{ID_COL}' not found in CSV columns")

df = df.copy()
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
df["__word_count__"] = df[TEXT_COL].str.split().map(len)

# basic filtering
df = df[df[TEXT_COL].str.strip().ne("")]
df = df[df["__word_count__"] >= MIN_WORDS]
df = df[df[TEXT_COL].str.len() >= MIN_TOKEN_LEN]

print("Rows after filtering:", len(df))
df.head(3)


Columns: ['doc_id', 'processed_v2']
Rows before filtering: 18846
Rows after filtering: 18846


,doc_id,processed_v2,__word_count__
0,0,from where s my thing subject what car is this nntp posting host rac3 wam umd edu organization university of maryland college park lines 15 i was wondering if anyone out there could enlighten me o...,127
1,1,from guy kuo subject si clock poll final call summary final call for si clock reports keywords si acceleration clock upgrade article i d shelley 1qvfo9innc3s organization university of washington ...,133
2,2,from thomas e willis subject pb questions organization purdue university engineering computer network distribution usa lines 36 well folks my mac plus finally gave up the ghost this weekend after ...,335


In [ ]:
# =========================
# 5. Sanity check and sample docs
# =========================
print("Average words:", round(df["__word_count__"].mean(), 2))
print("Median words:", float(df["__word_count__"].median()))
print("Min words:", int(df["__word_count__"].min()))
print("Max words:", int(df["__word_count__"].max()))

sample_show = df[[TEXT_COL]].sample(min(5, len(df)), random_state=42)
sample_show


Average words: 295.8
Median words: 174.0
Min words: 12
Max words: 20759


,processed_v2
18265,organization esoc european space operations centre from subject re nissan nomenclature was re manual shift bigots wanted lines 11 the european version is called 200 sx and have a 1 8 liter engine ...
423,from fletcher p adams subject pork c 17 c 5 was re abolish selective service oanization mississippi state university nntp posting host ra msstate edu organization mississippi state university line...
7900,from ron phillips subject randy weaver trial day 2 nntp posting host hound reply to organization intergraph electronics mountain view ca distribution usa lines 89 this was posted to the firearms p...
14785,from subject re homosexuality issues in christianity organization fnal ad net lines 39 in article michael siemon writes romans 1 27 i corinthians 6 9 i timothy 1 10 jude 1 7 ii peter 2 6 9 gen 19 ...
5217,from dick young subject attn h wheaton ucal davis nntp posting host 136 182 211 36 organization motorola inc cellular infrastructure distribution usa lines 11 i tried to e mail you but the message...


In [ ]:
# =========================
# 6. Vectorizers
# =========================
lsa_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=NGRAM_RANGE_LSA,
    min_df=MIN_DF,
    max_df=MAX_DF,
    lowercase=True,
)

lda_vectorizer = CountVectorizer(
    analyzer="word",
    ngram_range=NGRAM_RANGE_LDA,
    min_df=MIN_DF,
    max_df=MAX_DF,
    lowercase=True,
)

X_lsa = lsa_vectorizer.fit_transform(df[TEXT_COL])
X_lda = lda_vectorizer.fit_transform(df[TEXT_COL])

print("LSA matrix shape:", X_lsa.shape)
print("LDA matrix shape:", X_lda.shape)

lsa_vocab = np.array(lsa_vectorizer.get_feature_names_out())
lda_vocab = np.array(lda_vectorizer.get_feature_names_out())


LSA matrix shape: (18846, 315734)
LDA matrix shape: (18846, 45291)


In [ ]:
# =========================
# 7. Helper functions
# =========================
def get_top_words_for_component(component, vocab, top_n=10):
    top_idx = np.argsort(component)[::-1][:top_n]
    return [vocab[i] for i in top_idx]

def print_topics(model_name, topic_word_matrix, vocab, top_n=10):
    print(f"\n===== {model_name} =====")
    topics = []
    for topic_idx, component in enumerate(topic_word_matrix):
        top_words = get_top_words_for_component(component, vocab, top_n=top_n)
        topics.append((topic_idx, top_words))
        print(f"Topic {topic_idx}: {', '.join(top_words)}")
    return topics

def top_docs_for_topic(doc_topic_matrix, docs, topic_idx, top_n=2):
    scores = doc_topic_matrix[:, topic_idx]
    best_idx = np.argsort(scores)[::-1][:top_n]
    rows = []
    for i in best_idx:
        rows.append({
            "doc_index": int(i),
            "score": float(scores[i]),
            "text": docs.iloc[i][:600],
        })
    return pd.DataFrame(rows)

def topic_summary_df(topics):
    return pd.DataFrame(
        [{"topic": t, "top_words": ", ".join(words)} for t, words in topics]
    )


In [ ]:
lsa_results = {}

for k in LSA_K_LIST:
    print(f"\n### LSA, k={k}")
    svd = TruncatedSVD(n_components=k, random_state=42)
    doc_topic = svd.fit_transform(X_lsa)
    topic_word = svd.components_
    topics = print_topics(f"LSA k={k}", topic_word, lsa_vocab, top_n=10)
    lsa_results[k] = {
        "model": svd,
        "doc_topic": doc_topic,
        "topic_word": topic_word,
        "topics": topics,
        "explained_variance_sum": float(svd.explained_variance_ratio_.sum()),
    }
    print("Explained variance sum:", round(lsa_results[k]["explained_variance_sum"], 4))



### LSA, k=5

===== LSA k=5 =====
Topic 0: to, of, and, in, that, is, it, you, for, not
Topic 1: that, he, god, of, we, was, his, they, people, were
Topic 2: you, is, it, your, god, that, do, to, not, can
Topic 3: of, of the, and, armenian, by, key, turkish, armenians, government, encryption
Topic 4: he, god, his, is, windows, jesus, scsi, drive, him, card
Explained variance sum: 0.0127

### LSA, k=8

===== LSA k=8 =====
Topic 0: to, of, and, in, that, is, it, you, for, not
Topic 1: that, he, god, of, we, was, his, they, people, were
Topic 2: you, is, it, your, that, god, do, to, not, can
Topic 3: of, of the, and, armenian, by, key, turkish, armenians, government, encryption
Topic 4: he, god, his, windows, scsi, is, jesus, drive, dos, him
Topic 5: god, of, university, edu, posting, nntp, nntp posting, posting host, host, university of
Topic 6: you, scsi, drive, armenian, ide, armenians, turkish, were, and, my
Topic 7: gordon banks, banks, gordon, it, is, to gordon, banks n3jxp, n3jxp,

In [ ]:
lda_results = {}

for k in LDA_K_LIST:
    print(f"\n### LDA, k={k}")
    lda = LatentDirichletAllocation(
        n_components=k,
        random_state=42,
        learning_method="batch",
        max_iter=20,
        evaluate_every=-1,
    )
    doc_topic = lda.fit_transform(X_lda)
    topic_word = lda.components_
    topics = print_topics(f"LDA k={k}", topic_word, lda_vocab, top_n=10)
    lda_results[k] = {
        "model": lda,
        "doc_topic": doc_topic,
        "topic_word": topic_word,
        "topics": topics,
        "perplexity": float(lda.perplexity(X_lda)),
    }
    print("Perplexity:", round(lda_results[k]["perplexity"], 2))



### LDA, k=5

===== LDA k=5 =====
Topic 0: of, and, in, to, for, on, by, was, at, he
Topic 1: to, and, of, is, for, in, it, on, with, you
Topic 2: to, in, and, of, it, that, is, you, for, on
Topic 3: to, of, and, that, is, in, it, you, not, are
Topic 4: ax, max, key, clipper, encryption, g9v, b8f, a86, 145, chip
Perplexity: 1827.54

### LDA, k=8

===== LDA k=8 =====
Topic 0: to, and, of, for, is, in, on, or, this, file
Topic 1: for, 00, 10, card, drive, and, 25, dos, with, 16
Topic 2: car, bike, dod, engine, cx, cars, bmw, ride, ohio, ed
Topic 3: of, to, that, and, is, in, you, it, not, are
Topic 4: ax, max, g9v, b8f, a86, 145, pl, 1d9, 2di, 0t
Topic 5: of, to, and, in, that, is, for, be, it, on
Topic 6: to, it, is, of, you, and, in, that, for, have
Topic 7: in, to, and, of, he, was, for, that, on, is
Perplexity: 1772.5


In [ ]:
# Choose one configuration to inspect in detail
BEST_LSA_K = LSA_K_LIST[-1]
BEST_LDA_K = LDA_K_LIST[-1]

print("Inspecting LSA k =", BEST_LSA_K)
lsa_doc_topic = lsa_results[BEST_LSA_K]["doc_topic"]
for topic_idx, words in lsa_results[BEST_LSA_K]["topics"]:
    print(f"\n--- LSA Topic {topic_idx}: {', '.join(words)}")
    display(top_docs_for_topic(lsa_doc_topic, df[TEXT_COL], topic_idx, top_n=2))

print("\n\nInspecting LDA k =", BEST_LDA_K)
lda_doc_topic = lda_results[BEST_LDA_K]["doc_topic"]
for topic_idx, words in lda_results[BEST_LDA_K]["topics"]:
    print(f"\n--- LDA Topic {topic_idx}: {', '.join(words)}")
    display(top_docs_for_topic(lda_doc_topic, df[TEXT_COL], topic_idx, top_n=2))


Inspecting LSA k = 8

--- LSA Topic 0: to, of, and, in, that, is, it, you, for, not


,doc_index,score,text
0,13539,0.642220,from the white house subject clinton president s press conference 4 23 93 organization mit artificial intelligence lab lines 838 nntp posting host life ai mit edu the white house office of the pre...
1,14892,0.624211,from russell earl whitaker subject from crossbows to cryptography distribution world organization extropy institute reply to x mailer simple news 1 90 ka9q dis 1 19 lines 566 begin pgp signed mess...



--- LSA Topic 1: that, he, god, of, we, was, his, they, people, were


,doc_index,score,text
0,13163,0.226141,from paul fortmann pg subject praying for justice organization university of natal durban lines 650 i recently came across this article which i found interesting i have posted it to hear what othe...
1,16380,0.223093,from randy l hunt subject love in the morning by malcolm smith ministries lines 184 begin included message the following teaching is brought to you on behalf of malcolm smith ministries a ministry...



--- LSA Topic 2: you, is, it, your, that, god, do, to, not, can


...

,doc_index,score,text
0,14077,0.999751,from serdar argic subject news azerbaijan 5 19 22 organization zuma distribution world lines 500 in article f h miandoab writes the following news from turan news agency in baku azerbaijan is brou...
1,5478,0.999520,from serdar argic subject armenian scholars on the extermination of 2 5 million muslim people reply to serdar argic distribution world lines 288 in article eduard wiener writes why don t you post ...



--- LDA Topic 6: to, it, is, of, you, and, in, that, for, have


,doc_index,score,text
0,17754,0.996630,from kurt bollacker subject re challenge to microsoft supporters lines 40 nntp posting host bal1 mc duke edu x newsreader tin version 1 1 pl9 tim glauert wrote in article kurt bollacker writes tim...
1,16022,0.995906,from brett ferrell subject re challenge to microsoft supporters nntp posting host ant occ uc edu organization university of cincinnati lines 28 in article dave laudicina writes microsoft is the la...



--- LDA Topic 7: in, to, and, of, he, was, for, that, on, is


,doc_index,score,text
0,3313,0.997837,from daryl turner subject my predictions nntp posting host gibson cc umanitoba ca organization university of manitoba winnipeg manitoba canada lines 79 smythe division vancouver vs winnipeg jets i...
1,1868,0.997626,from tommi vartiainen subject re finland sweden vs nhl teams was helsinki stockholm nhl expansion nntp posting host vipunen hut fi organization helsinki university of technology finland lines 51 i...


In [ ]:
def dominant_topic_distribution(doc_topic_matrix):
    return np.bincount(doc_topic_matrix.argmax(axis=1), minlength=doc_topic_matrix.shape[1])

comparison_rows = []

for k, res in lsa_results.items():
    dist = dominant_topic_distribution(res["doc_topic"])
    comparison_rows.append({
        "model": "LSA",
        "k": k,
        "doc_topic_entropy_proxy": float(-(dist / dist.sum() * np.log((dist / dist.sum()) + 1e-12)).sum()),
        "explained_variance_sum": res["explained_variance_sum"],
    })

for k, res in lda_results.items():
    dist = dominant_topic_distribution(res["doc_topic"])
    comparison_rows.append({
        "model": "LDA",
        "k": k,
        "doc_topic_entropy_proxy": float(-(dist / dist.sum() * np.log((dist / dist.sum()) + 1e-12)).sum()),
        "perplexity": res["perplexity"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


,model,k,doc_topic_entropy_proxy,explained_variance_sum,perplexity
0,LSA,5,0.027192,0.012707,NaN
1,LSA,8,0.175486,0.016536,NaN
2,LDA,5,1.291067,NaN,1827.543114
3,LDA,8,1.570856,NaN,1772.498540
